# AlphaEvolve HCLS Demo: HP Protein Folding Visualization

This notebook visualizes the results of running AlphaEvolve to optimize a protein folding heuristic on a 2D square lattice.

The problem uses the **HP (Hydrophobic-Polar) model** where:
* **H** (Hydrophobic) monomers want to be close to each other to avoid water.
* **P** (Polar) monomers are comfortable facing water.
* The goal is to maximize **H-H contacts** (adjacent H monomers on the lattice that are not adjacent in the sequence).
* The fold must be **self-avoiding** (no two monomers share the same grid point) and consecutive monomers must be adjacent on the grid (distance = 1).

## Run Instructions

1. Make sure you have set up Application Default Credentials (ADC):
   ```bash
   gcloud auth application-default login
   ```
   *Note: If running in Google Colab, the "Colab Setup" cell below will handle authentication.*
2. Run the evolution from the `src` directory:
   ```bash
   cd src
   ../venv/bin/python run.py --problem_config=./data/problem_config/protein_folding.yaml
   ```
   This will run the evolution and write logs to `src/evolution_log.jsonl`.
3. Run the cells below to visualize the progress and the best found fold.

In [1]:
# @title Colab Setup
import os
import sys

# Detect if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # 1. Authenticate user for Vertex AI (ADC)
    print("Authenticating user for GCP...")
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated!")

    # 2. Clone the repository if not already present
    REPO_NAME = "alpha-evolve-demo"
    REPO_URL = f"https://github.com/goriri/{REPO_NAME}.git"
    
    if not os.path.exists(REPO_NAME):
        print(f"Cloning {REPO_URL}...")
        try:
            # Using system command to clone
            os.system(f"git clone {REPO_URL}")
            if os.path.exists(REPO_NAME):
                sys.path.append(os.path.abspath(REPO_NAME))
                os.chdir(REPO_NAME)
                print(f"Changed directory to {os.getcwd()}")
            else:
                print("Clone failed, directory not found.")
        except Exception as e:
            print(f"Failed to clone repository: {e}")
            print("Please upload the required files manually.")
    else:
        sys.path.append(os.path.abspath(REPO_NAME))
        os.chdir(REPO_NAME)
        print(f"Repository already exists. Changed directory to {os.getcwd()}")

    # 3. Install dependencies
    if os.path.exists("src/requirements.txt"):
        print("Installing dependencies...")
        os.system("pip install -r src/requirements.txt")
        print("Dependencies installed!")
    else:
        print("requirements.txt not found. Cannot install dependencies automatically.")
else:
    print("Running in local environment.")
    print("Ensure you have run 'gcloud auth application-default login' to set up ADC if you plan to run evolution.")

Running in local environment.
Ensure you have run 'gcloud auth application-default login' to set up ADC if you plan to run evolution.


In [2]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np

# Try to load the live run log first, fallback to demo log
log_path = 'src/evolution_log.jsonl'
demo_log_path = 'src/evolution_log_demo.jsonl'

def load_logs(path):
    if not os.path.exists(path):
        return None
    
    logs = []
    with open(path, 'r') as f:
        for line in f:
            if line.strip():
                logs.append(json.loads(line))
    return logs

logs = load_logs(log_path)
if logs is not None:
    print(f"Loaded {len(logs)} log entries from live run ({log_path}).")
else:
    logs = load_logs(demo_log_path)
    if logs is not None:
        print(f"Loaded {len(logs)} log entries from pre-run DEMO ({demo_log_path}).")
        print("Note: Run the evolution to generate your own results.")
    else:
        print("No log files found. Please run the evolution.")
        logs = []

Loaded 10 log entries from live run (src/evolution_log.jsonl).


In [3]:
if logs:
    steps = [entry['step'] for entry in logs]
    
    scores = []
    best_scores = []
    for entry in logs:
        if entry['status'] == 'success':
            scores.append(entry['scores'].get('hh_contacts', 0))
        else:
            scores.append(None)
        
        best_scores.append(entry['best_scores'].get('scores_to_optimize/hh_contacts', 0))
        
    plt.figure(figsize=(10, 5))
    plt.plot(steps, best_scores, label='Best Score (Cumulative)', color='green', linewidth=2)
    plt.scatter([s for s, sc in zip(steps, scores) if sc is not None], 
                [sc for sc in scores if sc is not None], 
                alpha=0.5, label='Individual Trial Score', color='blue', s=10)
    
    plt.xlabel('Step (LLM Call)')
    plt.ylabel('H-H Contacts')
    plt.title('AlphaEvolve Optimization Progress')
    plt.legend()
    plt.grid(True)
    plt.savefig('progress.png')
    plt.close()
else:
    print("No logs to plot.")

In [4]:
if logs:
    successful_entries = [e for e in logs if e['status'] == 'success']
    if successful_entries:
        best_entry = max(successful_entries, key=lambda e: e['scores'].get('hh_contacts', 0))
        best_code = best_entry['program']
        best_score = best_entry['scores'].get('hh_contacts', 0)
        print(f"Best score found: {best_score} at step {best_entry['step']}")
        
        exec_globals = globals().copy()
        try:
            exec(best_code, exec_globals)
            fold_protein_fn = exec_globals.get('fold_protein')
        except Exception as e:
            print("Failed to execute best program code:", e)
            fold_protein_fn = None
            
        if fold_protein_fn:
            sequence = "HPHPPHHPHPPHHPPHPHHP"
            try:
                coords = fold_protein_fn(sequence)
                coords = [(int(x), int(y)) for x, y in coords]
                print("Coordinates:", coords)
                
                xs = [c[0] for c in coords]
                ys = [c[1] for c in coords]
                
                plt.figure(figsize=(8, 8))
                plt.plot(xs, ys, color='gray', linestyle='-', linewidth=2, zorder=1)
                
                h_xs, h_ys = [], []
                p_xs, p_ys = [], []
                for i, (x, y) in enumerate(coords):
                    if sequence[i] == 'H':
                        h_xs.append(x)
                        h_ys.append(y)
                    else:
                        p_xs.append(x)
                        p_ys.append(y)
                        
                plt.scatter(h_xs, h_ys, color='blue', s=200, label='H (Hydrophobic)', zorder=2)
                plt.scatter(p_xs, p_ys, color='red', s=200, label='P (Polar)', zorder=2)
                
                for i, (x, y) in enumerate(coords):
                    plt.annotate(str(i), (x, y), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold')
                
                n = len(sequence)
                for i in range(n):
                    if sequence[i] == 'H':
                        for j in range(i + 2, n):
                            if sequence[j] == 'H':
                                p1 = coords[i]
                                p2 = coords[j]
                                dist = abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])
                                if dist == 1:
                                    plt.plot([p1[0], p2[0]], [p1[1], p2[1]], color='green', linestyle='--', linewidth=2, label='H-H Contact' if 'H-H Contact' not in plt.gca().get_legend_handles_labels()[1] else "")
                
                plt.title(f'Best Fold (H-H Contacts: {best_score})')
                plt.legend()
                plt.grid(True)
                plt.gca().set_aspect('equal', adjustable='box')
                plt.savefig('best_fold.png')
                plt.close()
                
            except Exception as e:
                print("Failed to run fold_protein or plot:", e)
        else:
            print("fold_protein function not found in best code.")
    else:
        print("No successful steps in logs.")
else:
    print("No logs available.")

Best score found: 6.0 at step 6
Coordinates: [(0, 0), (1, 0), (1, -1), (0, -1), (0, -2), (1, -2), (2, -2), (2, -1), (2, 0), (3, 0), (3, 1), (2, 1), (1, 1), (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (2, 2), (3, 2)]
